# 🚀 CLIP Embedder for Google Colab (A100 Multi-GPU)

Bu notebook, CLIP embedding'lerini Google Colab'da çalıştırmak için tasarlanmıştır.

**Özellikler:**
- 🎯 Multi-process: 4-5 CLIP modeli paralel çalışır (A100 için optimize)
- 💾 Google Drive'a otomatik kayıt
- 🔄 Checkpoint/Resume: Kesintilerde kaldığı yerden devam
- 📁 Klasör bazlı işleme: Her klasörden max 5000 resim

**Kullanım:**
1. Drive'ı bağla
2. Ayarları düzenle (input/output path)
3. Tüm hücreleri çalıştır

In [ ]:
# 📦 Gerekli paketleri yükle
!pip install -q open-clip-torch pandas pyarrow tqdm

## 📁 Adım 1: Drive Mount & Veri Hazırlama

Zip dosyalarını Drive'dan Colab disk'e kopyala ve unzip yap.

In [1]:
# 🔗 Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 📂 Drive'daki zip dosyalarını listele
import os
from pathlib import Path

# Drive'daki zip klasörünü belirt
DRIVE_ZIP_DIR = "/content/drive/MyDrive/furniture-retrieval"  # <- Kendi path'ini yaz

# Zip dosyalarını listele
if os.path.exists(DRIVE_ZIP_DIR):
    zip_files = [f for f in os.listdir(DRIVE_ZIP_DIR) if f.endswith('.zip')]
    print(f"📦 Bulunan zip dosyaları ({len(zip_files)}):")
    for zf in sorted(zip_files):
        size_mb = os.path.getsize(os.path.join(DRIVE_ZIP_DIR, zf)) / 1e6
        print(f"   {zf} ({size_mb:.1f} MB)")
else:
    print(f"⚠️ Klasör bulunamadı: {DRIVE_ZIP_DIR}")

📦 Bulunan zip dosyaları (4):
   images-part1.zip (37979.0 MB)
   images-part2.zip (40165.1 MB)
   images-part3.zip (53296.2 MB)
   outputs_requests(not_resized).zip (29.8 MB)


In [3]:
# 📁 Çalışma klasörlerini oluştur
LOCAL_DATA_DIR = "/content/data"           # Zip'lerin açılacağı yer
LOCAL_IMAGES_DIR = "/content/data/images"  # Resimlerin olacağı yer (embedder input)

!mkdir -p {LOCAL_DATA_DIR}
!mkdir -p {LOCAL_IMAGES_DIR}

print(f"✅ Klasörler oluşturuldu:")
print(f"   📁 Data: {LOCAL_DATA_DIR}")
print(f"   📁 Images: {LOCAL_IMAGES_DIR}")

✅ Klasörler oluşturuldu:
   📁 Data: /content/data
   📁 Images: /content/data/images


In [ ]:
# 📦 Tek bir zip dosyasını kopyala ve aç
# Bu hücreyi her zip için manuel çalıştır

ZIP_NAME = "images-part2.zip"  # <- Hangi zip'i açmak istiyorsan yaz

import shutil

src_path = os.path.join(DRIVE_ZIP_DIR, ZIP_NAME)
dst_path = os.path.join(LOCAL_DATA_DIR, ZIP_NAME)

if os.path.exists(src_path):
    print(f"📋 Kopyalanıyor: {ZIP_NAME}")
    shutil.copy2(src_path, dst_path)
    print(f"✅ Kopyalandı: {dst_path}")

    # Unzip
    print(f"📦 Açılıyor...")
    !unzip -q -o {dst_path} -d {LOCAL_IMAGES_DIR}
    print(f"✅ Açıldı!")

    # Zip'i sil (disk tasarrufu)
    os.remove(dst_path)
    print(f"🗑️ Zip silindi (disk tasarrufu)")
else:
    print(f"⚠️ Dosya bulunamadı: {src_path}")

📋 Kopyalanıyor: images-part2.zip


In [ ]:
# 🔍 Açılan klasör yapısını incele
# Bazı zip'ler düz resim, bazıları iç içe klasör olabilir

print(f"📁 {LOCAL_IMAGES_DIR} içeriği:\n")
!ls -la {LOCAL_IMAGES_DIR}

print(f"\n📊 Klasör yapısı (ilk 3 seviye):")
!find {LOCAL_IMAGES_DIR} -maxdepth 3 -type d | head -30

### 🔧 Preprocess - Klasör Yapısını Düzelt

Bazı zip'ler açılınca şu yapıda olabilir:
- `images/folder1/folder1/img.jpg` (iç içe)
- `images/folder1.zip_extracted/img.jpg` (garip isim)
- `images/folder1/subfolder/img.jpg` (alt klasör)

Aşağıdaki hücrelerle düzeltebilirsin:

In [ ]:
# 🔧 PREPROCESS FONKSİYONLARI

import shutil
from pathlib import Path

def flatten_nested_folder(folder_path: str, folder_name: str = None):
    """
    İç içe klasörü düzleştir.
    Örn: images/folder1/folder1/*.jpg -> images/folder1/*.jpg
    """
    folder = Path(folder_path)
    if folder_name is None:
        folder_name = folder.name

    nested = folder / folder_name
    if nested.exists() and nested.is_dir():
        print(f"🔧 İç içe klasör bulundu: {nested}")
        # İçerideki dosyaları dışarı taşı
        for item in nested.iterdir():
            dest = folder / item.name
            if not dest.exists():
                shutil.move(str(item), str(dest))
        # Boş klasörü sil
        if not any(nested.iterdir()):
            nested.rmdir()
            print(f"✅ Düzleştirildi: {folder}")
    else:
        print(f"ℹ️ İç içe klasör yok: {folder}")


def flatten_all_subfolders(folder_path: str):
    """
    Tüm alt klasörlerdeki resimleri ana klasöre taşı.
    Örn: images/folder1/sub1/*.jpg + images/folder1/sub2/*.jpg -> images/folder1/*.jpg
    """
    folder = Path(folder_path)
    image_exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tiff'}

    moved = 0
    for subdir in folder.rglob('*'):
        if subdir.is_file() and subdir.suffix.lower() in image_exts:
            if subdir.parent != folder:
                dest = folder / subdir.name
                # Aynı isimde varsa numara ekle
                if dest.exists():
                    stem = dest.stem
                    suffix = dest.suffix
                    counter = 1
                    while dest.exists():
                        dest = folder / f"{stem}_{counter}{suffix}"
                        counter += 1
                shutil.move(str(subdir), str(dest))
                moved += 1

    # Boş klasörleri sil
    for subdir in sorted(folder.rglob('*'), reverse=True):
        if subdir.is_dir() and not any(subdir.iterdir()):
            subdir.rmdir()

    print(f"✅ {moved} resim taşındı, alt klasörler temizlendi")


def rename_folder(old_name: str, new_name: str):
    """Klasör adını değiştir."""
    old_path = Path(LOCAL_IMAGES_DIR) / old_name
    new_path = Path(LOCAL_IMAGES_DIR) / new_name

    if old_path.exists():
        old_path.rename(new_path)
        print(f"✅ Yeniden adlandırıldı: {old_name} -> {new_name}")
    else:
        print(f"⚠️ Klasör bulunamadı: {old_name}")


print("✅ Preprocess fonksiyonları yüklendi")

In [ ]:
# 🔧 ÖRNEK: İç içe klasörü düzleştir
# flatten_nested_folder(f"{LOCAL_IMAGES_DIR}/folder1", "folder1")

# 🔧 ÖRNEK: Tüm alt klasörleri ana klasöre taşı
# flatten_all_subfolders(f"{LOCAL_IMAGES_DIR}/folder1")

# 🔧 ÖRNEK: Klasör adını değiştir
# rename_folder("folder1.zip_extracted", "folder1")

# ⬇️ Buraya kendi preprocess komutlarını yaz:
# -------------------------------------------------



In [ ]:
# ✅ Final klasör yapısını kontrol et
print(f"📁 {LOCAL_IMAGES_DIR} son durumu:\n")

from pathlib import Path

for folder in sorted(Path(LOCAL_IMAGES_DIR).iterdir()):
    if folder.is_dir():
        # Resim sayısını bul
        image_count = sum(1 for f in folder.rglob('*')
                         if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif'})
        # Alt klasör var mı?
        subdirs = [d for d in folder.iterdir() if d.is_dir()]

        status = "✅" if not subdirs else f"⚠️ ({len(subdirs)} alt klasör)"
        print(f"   {status} {folder.name}: {image_count} resim")

---
## 🚀 Adım 2: Embedding Çıkar

Klasör yapısı hazır olduğunda aşağıdaki hücreleri çalıştır.

In [ ]:
# ⚙️ AYARLAR
# ============================================

# Resimlerin bulunduğu ana klasör (Colab disk - hızlı)
INPUT_DIR = "/content/data/images"

# Embedding'lerin kaydedileceği klasör (Drive - kalıcı)
OUTPUT_DIR = "/content/drive/MyDrive/datasets/embeddings"

# Klasör başına maksimum resim sayısı
MAX_IMAGES_PER_FOLDER = 5000

# Batch boyutu (A100 için 128-256 iyi çalışır)
BATCH_SIZE = 128

# Paralel worker sayısı (A100 80GB için 4-5, 40GB için 2-3)
NUM_WORKERS = 4

# CLIP model adı
MODEL_NAME = "ViT-B-32"

print(f"📁 Input (local disk): {INPUT_DIR}")
print(f"📁 Output (Drive): {OUTPUT_DIR}")
print(f"🔢 Max per folder: {MAX_IMAGES_PER_FOLDER}")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"👥 Workers: {NUM_WORKERS}")

In [ ]:
# 📚 Import'lar ve Yardımcı Fonksiyonlar
import os
import json
import random
from pathlib import Path
from typing import List, Tuple, Optional, Set
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
from queue import Queue
import threading

import torch
import torch.multiprocessing as torch_mp
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import pandas as pd
import open_clip

# GPU kontrolü
print(f"🎮 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tiff'}
CHECKPOINT_FILE = "checkpoint.json"

In [ ]:
# 🔧 Yardımcı Fonksiyonlar

def get_image_paths(folder: Path) -> List[Path]:
    """Klasördeki tüm resim dosyalarını bul."""
    images = []
    for ext in IMAGE_EXTENSIONS:
        images.extend(folder.glob(f'*{ext}'))
        images.extend(folder.glob(f'*{ext.upper()}'))
    return images


def save_checkpoint(output_dir: Path, state: dict):
    """Checkpoint kaydet."""
    checkpoint_path = output_dir / CHECKPOINT_FILE
    state['last_updated'] = datetime.now().isoformat()
    with open(checkpoint_path, 'w') as f:
        json.dump(state, f, indent=2)
    print(f"💾 Checkpoint: {len(state.get('completed_folders', []))} klasör tamamlandı")


def load_checkpoint(output_dir: Path) -> Optional[dict]:
    """Varsa checkpoint'i yükle."""
    checkpoint_path = output_dir / CHECKPOINT_FILE
    if not checkpoint_path.exists():
        return None
    try:
        with open(checkpoint_path) as f:
            state = json.load(f)
        print(f"📂 Checkpoint bulundu: {state.get('last_updated', 'unknown')}")
        return state
    except:
        return None


def get_completed_folders(output_dir: Path) -> Set[str]:
    """Zaten tamamlanmış klasörleri bul."""
    checkpoint = load_checkpoint(output_dir)
    if checkpoint:
        return set(checkpoint.get('completed_folders', []))
    return set()


print("✅ Yardımcı fonksiyonlar yüklendi")

In [ ]:
# 🧠 CLIP Worker - Paralel Embedding İşlemi

class CLIPWorker:
    """Tek bir CLIP modeli ile embedding çıkaran worker."""

    def __init__(self, worker_id: int, model_name: str = "ViT-B-32", device: str = "cuda"):
        self.worker_id = worker_id
        self.device = device
        self.model_name = model_name
        self.model = None
        self.preprocess = None

    def load_model(self):
        """Modeli yükle."""
        if self.model is None:
            self.model, _, self.preprocess = open_clip.create_model_and_transforms(
                self.model_name,
                pretrained='laion2b_s34b_b79k'
            )
            self.model = self.model.to(self.device).eval()
        return self

    def process_batch(self, image_paths: List[str]) -> Tuple[List[str], np.ndarray]:
        """Bir batch resmi işle."""
        if self.model is None:
            self.load_model()

        valid_paths = []
        batch_images = []

        for path in image_paths:
            try:
                img = Image.open(path).convert('RGB')
                img = self.preprocess(img)
                batch_images.append(img)
                valid_paths.append(path)
            except Exception as e:
                continue

        if not batch_images:
            return [], np.array([])

        with torch.no_grad():
            images_tensor = torch.stack(batch_images).to(self.device)
            features = self.model.encode_image(images_tensor)
            features = features / features.norm(dim=-1, keepdim=True)
            features = features.cpu().numpy()

        return valid_paths, features


def process_folder_parallel(
    folder_path: Path,
    output_dir: Path,
    max_images: int,
    batch_size: int,
    num_workers: int,
    model_name: str
) -> Tuple[int, str]:
    """
    Bir klasörü paralel olarak işle.
    Resimleri worker'lara dağıt, sonuçları birleştir.
    """
    folder_name = folder_path.name

    # Resimleri bul
    image_paths = get_image_paths(folder_path)

    if len(image_paths) == 0:
        return 0, folder_name

    # Max sayıya göre rastgele seç
    if len(image_paths) > max_images:
        random.seed(42 + hash(folder_name) % 10000)
        image_paths = random.sample(image_paths, max_images)

    image_paths = [str(p) for p in image_paths]

    # Batch'lere böl
    batches = [image_paths[i:i+batch_size] for i in range(0, len(image_paths), batch_size)]

    # Worker'ları oluştur
    workers = [CLIPWorker(i, model_name).load_model() for i in range(min(num_workers, len(batches)))]

    all_paths = []
    all_embeddings = []

    # Progress bar
    pbar = tqdm(total=len(batches), desc=f"📁 {folder_name}", leave=False)

    # Round-robin ile batch'leri dağıt
    for batch_idx, batch in enumerate(batches):
        worker = workers[batch_idx % len(workers)]
        paths, embeddings = worker.process_batch(batch)

        if len(paths) > 0:
            all_paths.extend(paths)
            all_embeddings.append(embeddings)

        pbar.update(1)

    pbar.close()

    if not all_embeddings:
        return 0, folder_name

    # Birleştir
    combined_embeddings = np.vstack(all_embeddings)

    # Kaydet
    df = pd.DataFrame({
        'image_path': all_paths,
        'folder': folder_name
    })

    embeddings_file = output_dir / f"{folder_name}_embeddings.npy"
    metadata_file = output_dir / f"{folder_name}_metadata.parquet"

    np.save(embeddings_file, combined_embeddings)
    df.to_parquet(metadata_file, index=False)

    # Worker'ları temizle (GPU memory)
    for worker in workers:
        del worker.model
        del worker.preprocess
    torch.cuda.empty_cache()

    return len(all_paths), folder_name


print("✅ CLIP Worker sınıfı yüklendi")

In [ ]:
# 🚀 Ana İşlem Fonksiyonu

def run_embedder(
    input_dir: str,
    output_dir: str,
    max_per_folder: int = 5000,
    batch_size: int = 128,
    num_workers: int = 4,
    model_name: str = "ViT-B-32",
    resume: bool = True
):
    """
    Tüm klasörleri işle ve embedding'leri Drive'a kaydet.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    # Output klasörünü oluştur
    output_path.mkdir(parents=True, exist_ok=True)

    # Alt klasörleri bul
    folders = sorted([d for d in input_path.iterdir() if d.is_dir()], key=lambda x: x.name)

    if not folders:
        folders = [input_path]

    print(f"📁 Bulunan klasör sayısı: {len(folders)}")

    # Checkpoint kontrolü
    completed_folders = set()
    if resume:
        completed_folders = get_completed_folders(output_path)
        if completed_folders:
            print(f"✅ Daha önce tamamlanan: {len(completed_folders)} klasör")

    # Summary
    summary = {
        "folders": {},
        "total_images": 0,
        "embedding_dim": 512
    }

    summary_file = output_path / "summary.json"
    if summary_file.exists() and resume:
        try:
            with open(summary_file) as f:
                summary = json.load(f)
            print(f"📂 Önceki summary: {summary['total_images']} resim")
        except:
            pass

    # State
    state = {
        "completed_folders": list(completed_folders),
        "total_folders": len(folders)
    }

    # Ana progress bar
    main_pbar = tqdm(total=len(folders), desc="🚀 Toplam İlerleme")
    main_pbar.update(len(completed_folders))

    for folder in folders:
        folder_name = folder.name

        # Zaten tamamlandıysa atla
        if folder_name in completed_folders:
            continue

        print(f"\n{'='*50}")
        print(f"📁 Klasör: {folder_name}")

        # Klasörü işle
        count, name = process_folder_parallel(
            folder_path=folder,
            output_dir=output_path,
            max_images=max_per_folder,
            batch_size=batch_size,
            num_workers=num_workers,
            model_name=model_name
        )

        if count > 0:
            print(f"✅ {count} resim işlendi")

            summary["folders"][folder_name] = {
                "count": count,
                "embeddings_file": str(output_path / f"{folder_name}_embeddings.npy"),
                "metadata_file": str(output_path / f"{folder_name}_metadata.parquet")
            }
            summary["total_images"] += count
        else:
            print(f"⚠️ Resim bulunamadı veya işlenemedi")

        # Checkpoint kaydet
        state["completed_folders"].append(folder_name)
        save_checkpoint(output_path, state)

        # Summary güncelle
        with open(summary_file, 'w') as f:
            json.dump(summary, f, indent=2)

        main_pbar.update(1)

        # GPU memory temizle
        torch.cuda.empty_cache()

    main_pbar.close()

    print(f"\n{'='*50}")
    print(f"✅ TAMAMLANDI!")
    print(f"📊 Toplam işlenen resim: {summary['total_images']}")
    print(f"📁 Çıktı klasörü: {output_path}")

    # Checkpoint temizle
    checkpoint_path = output_path / CHECKPOINT_FILE
    if checkpoint_path.exists():
        checkpoint_path.unlink()
        print("🧹 Checkpoint temizlendi")

    return summary


print("✅ Ana işlem fonksiyonu yüklendi")

## ▶️ Çalıştır

Aşağıdaki hücreyi çalıştırarak embedding işlemini başlat.
Kesinti olursa tekrar çalıştır - kaldığı yerden devam eder.

In [ ]:
# ▶️ ÇALIŞTIR
summary = run_embedder(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    max_per_folder=MAX_IMAGES_PER_FOLDER,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    model_name=MODEL_NAME,
    resume=True  # Kesintiden sonra devam et
)

## 📊 Sonuçları İncele

In [ ]:
# 📊 Summary'yi görüntüle
output_path = Path(OUTPUT_DIR)
summary_file = output_path / "summary.json"

if summary_file.exists():
    with open(summary_file) as f:
        summary = json.load(f)

    print(f"📊 ÖZET")
    print(f"{'='*50}")
    print(f"Toplam resim: {summary['total_images']}")
    print(f"Klasör sayısı: {len(summary['folders'])}")
    print(f"\n📁 Klasör dağılımı:")
    for folder, info in sorted(summary['folders'].items()):
        print(f"   {folder}: {info['count']} resim")
else:
    print("⚠️ Summary dosyası bulunamadı. Önce embedder'ı çalıştırın.")

In [ ]:
# 🔍 Örnek embedding'leri kontrol et
output_path = Path(OUTPUT_DIR)

# İlk klasörün embedding'lerini yükle
npy_files = list(output_path.glob("*_embeddings.npy"))
if npy_files:
    sample_file = npy_files[0]
    embeddings = np.load(sample_file)
    print(f"📁 Dosya: {sample_file.name}")
    print(f"📐 Shape: {embeddings.shape}")
    print(f"📊 Embedding örneği (ilk 10 değer):")
    print(f"   {embeddings[0][:10]}")
else:
    print("⚠️ Embedding dosyası bulunamadı.")